# CISA Known Exploited Vulnerabilities (KEV) Machine Learning
Data source: The Cybersecurity and Infrastructure Security Agency's (CISA) [Known Exploited Vulnerabilities (KEV) catalog](https://github.com/cisagov/kev-data?)

## Background
I previously performed an exploratory data analysis on the KEV dataset to identify trends in known exploited vulnerabilities, remediation-timelines, and ransomware-associated weaknesses. 

My findings were as follows:
- Microsoft has the largest number of cataloged vulnerabilities.
- Input validation and use-after-free are the most common weaknesses.
- Ransomware vulnerabilities show similar remediation timelines to the broader catalog.
- Most remediation deadlines are exactly 21 days.
- Vulnerability additions peaked in 2022.

The best use-case for a regression model for this project would be estimating the remediation deadline of a particular vulnerability. However, my analysis project revealed dramatic spikes in the remediation days distribution at 14, 21, and 181 days. These results suggest that CISA uses standardized deadlines. Data that can be categorized in this fashion is better suited for a classification model than a regression model. 
However, there was enough variation among vulnerabilities in general versus ransomware-associated CWEs that a classification model might be able to successfully predict ransomware status. I am more excited by the prospect of classifying ransomware status than I am by classifying remediation deadlines. Therefore, I will train my model to do the former. 

## Step 1: Import Libraries

In [5]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

## Step 2: Load the Data

In [6]:
# Get the .csv data from GitHub
data_url = "https://raw.githubusercontent.com/cisagov/kev-data/refs/heads/develop/known_exploited_vulnerabilities.csv"
kev = pd.read_csv(data_url)

# Save a local copy
kev.to_csv("known_exploited_vulnerabilities.csv", index=False)
kev = pd.read_csv("known_exploited_vulnerabilities.csv")

print(kev.head(30))

             cveID    vendorProject  \
0   CVE-2026-16232      Check Point   
1   CVE-2026-50522        Microsoft   
2   CVE-2026-60137        WordPress   
3   CVE-2026-63030        WordPress   
4    CVE-2026-0770         Langflow   
5   CVE-2021-27137           DD-WRT   
6   CVE-2026-58644        Microsoft   
7   CVE-2026-25089         Fortinet   
8   CVE-2026-39808         Fortinet   
9   CVE-2026-46817           Oracle   
10   CVE-2023-4346  KNX Association   
11  CVE-2026-56155        Microsoft   
12  CVE-2026-56164        Microsoft   
13  CVE-2026-15409        SonicWall   
14  CVE-2026-15410        SonicWall   
15   CVE-2008-4128            Cisco   
16  CVE-2026-56291          Balbooa   
17  CVE-2026-48939         iCagenda   
18  CVE-2026-48908       JoomShaper   
19  CVE-2026-55255         Langflow   
20  CVE-2026-56290         Joomlack   
21  CVE-2026-48282            Adobe   
22  CVE-2026-45659        Microsoft   
23  CVE-2026-48558      SimpleHelp    
24  CVE-2026-12569       

## Step 3: Read the Data Documentation 
### Shema
#### (Extracted from known_exploited_vulnerabilities_schema.json)

| Column | Description |
| :--- | ---: | 
| cveID | The CVE ID of the vulnerability in the format CVE-YYYY-NNNN, note that the number portion can have more than 4 digits |
| vendorProject | The vendor or project name for the vulnerability |
| product | The vulnerability product |
| vulnerabilityName | The name of the vulnerability |
| dateAdded | The date the vulnerability was added to the catalog in the format YYYY-MM-DD |
| shortDescription | A short description of the vulnerability |
| requiredAction | The required action to address the vulnerability |
| dueDate | The date the required action is due in the format YYYY-MM-DD |
| knownRansomwareCampaignUse | 'Known' if this vulnerability is known to have been leveraged as part of a ransomware campaign; 'Unknown' if CISA lacks confirmation that the vulnerability has been utilized for ransomware |
| notes | Any additional notes about the vulnerability |
| cwes | Common Weakness Enumeration (CWE) codes associated with this vulnerability. CWEs are in the format CWE-NNNN; note that the number portion can have any number of digits |


## Step 4: Inspect the Data

In [7]:
print(kev.info())
print()
print()
print(kev.describe(include="all"))
print()
print()
kev.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1653 entries, 0 to 1652
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   cveID                       1653 non-null   str  
 1   vendorProject               1653 non-null   str  
 2   product                     1653 non-null   str  
 3   vulnerabilityName           1653 non-null   str  
 4   dateAdded                   1653 non-null   str  
 5   shortDescription            1653 non-null   str  
 6   requiredAction              1653 non-null   str  
 7   dueDate                     1653 non-null   str  
 8   knownRansomwareCampaignUse  1653 non-null   str  
 9   notes                       1653 non-null   str  
 10  cwes                        1482 non-null   str  
dtypes: str(11)
memory usage: 142.2 KB
None


                 cveID vendorProject  product  \
count             1653          1653     1653   
unique            1653           276      669

np.int64(0)

## Step 5: Clean the Data
Before analysis, I cleaned the dataset by renaming columns to snake_case and converting date columns to datetime objects.

In [8]:
# Rename columns to match Python's snake_case convention
kev.rename(columns={
    "cveID": "cve_id",
    "vendorProject": "vendor_project",
    "vulnerabilityName": "vulnerability_name",
    "dateAdded": "date_added",
    "shortDescription": "short_description",
    "requiredAction": "required_action",
    "dueDate": "due_date",
    "knownRansomwareCampaignUse": "known_ransomware_campaign_use"
}, inplace=True)

#print(kev.columns)



# Covert values in date_added and due_date from strings to dates
kev["date_added"] = pd.to_datetime(kev["date_added"])
kev["due_date"] = pd.to_datetime(kev["due_date"])

#print(kev.info())

## Step 6: Choosing the Model

I already decided on a classification model for this project. Below is a table detailing my assessment of the different classification techniques I could use. 


| Model | Is Appropriate? | Explanation |
| :--- | --- | ---: | 
| Logistic Regression | Appropriate | Excellent baseline, interpretable |
| Decision Forest (Random Forest) | Appropriate | Usually very strong on tabular data |
| Support Vector Machine (SVM) | Appropriate | Good for complex decision boundaries |
| Decision Tree | Has potential | Good for visualization, but often outperformed by forests |
| Naive Bayes | Has potential | Fast, but assumptions are often unrealistic |
| k-Nearest Neighbors | Inappropriate | Sensitive to scaling and many categorical variables |

### Logistic Regression

In [9]:
from sklearn.linear_model import LogisticRegression